In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
PDF_PATH = PROJECT_ROOT / "data" / "customer_support.pdf"
CHROMA_PATH = PROJECT_ROOT / "data" / "chroma_db"

print("Project root:", PROJECT_ROOT)
print("PDF path:", PDF_PATH)
print("PDF exists:", PDF_PATH.exists())
print("Chroma path:", CHROMA_PATH)

Project root: /home/workspace/learning/AI_Support_Eng
PDF path: /home/workspace/learning/AI_Support_Eng/data/customer_support.pdf
PDF exists: True
Chroma path: /home/workspace/learning/AI_Support_Eng/data/chroma_db


In [5]:
from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


In [6]:
reader = PdfReader(PDF_PATH)

print("Number of pages:", len(reader.pages))

Number of pages: 20


In [7]:
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    pages.append(
        {
            "page_number": page_number,
            "text": text,
        }
    )

print("Pages extracted:", len(pages))
print("First page characters:", len(pages[0]["text"]))

Pages extracted: 20
First page characters: 292


## 4. Inspect the extracted text

This step is easy to skip, but it is extremely important.

If PDF extraction is broken, everything after this point will also be broken. RAG quality cannot compensate for garbage input.

In [8]:
print(pages[0]["text"][:5000])

AcmeCloud
 
Support
 
&
 
Operations
 
Handbook
 
Document
 
Version:
 
3.2
 
 
Last
 
Updated:
 
September
 
2026
 
 
Document
 
Owner:
 
Customer
 
Operations
 
Team
 
 
Classification:
 
Internal
 
Support
 
Documentation
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 


### Inspect every page briefly

We want to know whether headings, paragraphs, and page boundaries survived extraction.

In [9]:
for page in pages:
    preview = page["text"].replace("\n", " ")[:180]
    print(f"Page {page['page_number']:>2}: {preview}...")

Page  1: AcmeCloud   Support   &   Operations   Handbook   Document   Version:   3.2     Last   Updated:   September   2026     Document   Owner:   Customer   Operations   Team     Classifi...
Page  2: Introduction   &   Customer   Support   Overview   1.   About   AcmeCloud   AcmeCloud   is   a   cloud-based   productivity   and   collaboration   platform   designed   for   indi...
Page  3: ●   Usage   information   General   product   information   does   not   require   customer   verification.   For   example:   "What   is   AcmeCloud's   refund   policy?"   is   a...
Page  4: Subscription   Plans   4.   AcmeCloud   Plans   AcmeCloud   offers   four   subscription   plans.   Free   The   Free   plan   is   intended   for   individuals   evaluating   Acme...
Page  5: Included:   ●   Up   to   20   team   members   ●   500   GB   storage   ●   Advanced   analytics   ●   API   access   ●   Advanced   reporting   ●   Priority   support   Professio...
Page  6: However,   if   the   cust

## 5. Convert pages into LangChain Documents

LangChain uses a `Document` object containing:

- `page_content` — the actual text.
- `metadata` — information about where that text came from.

Metadata becomes very useful later when we want to show citations such as **page 4 of the support handbook**.

In [10]:
documents = [
    Document(
        page_content=page["text"],
        metadata={
            "source": PDF_PATH.name,
            "page": page["page_number"],
        },
    )
    for page in pages
]

print("Documents:", len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:500])

Documents: 20
{'source': 'customer_support.pdf', 'page': 1}
AcmeCloud
 
Support
 
&
 
Operations
 
Handbook
 
Document
 
Version:
 
3.2
 
 
Last
 
Updated:
 
September
 
2026
 
 
Document
 
Owner:
 
Customer
 
Operations
 
Team
 
 
Classification:
 
Internal
 
Support
 
Documentation
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 


## 7. Create the text splitter

We will start with `RecursiveCharacterTextSplitter`.

`chunk_size=1000` means we aim for chunks around 1000 characters.

`chunk_overlap=150` means neighboring chunks can share some text. This reduces the chance that an important sentence is split exactly at a chunk boundary.

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

## 8. Split the documents into chunks

In [12]:
chunks = splitter.split_documents(documents)

print("Original documents:", len(documents))
print("Total chunks:", len(chunks))

Original documents: 20
Total chunks: 35


## 9. Inspect the chunks

Do not trust the splitter blindly. Look at the actual chunks.

In [13]:
for index, chunk in enumerate(chunks[:10]):
    print("=" * 80)
    print("Chunk:", index)
    print("Metadata:", chunk.metadata)
    print("Characters:", len(chunk.page_content))
    print(chunk.page_content[:1000])
    print()

Chunk: 0
Metadata: {'source': 'customer_support.pdf', 'page': 1}
Characters: 224
AcmeCloud
 
Support
 
&
 
Operations
 
Handbook
 
Document
 
Version:
 
3.2
 
 
Last
 
Updated:
 
September
 
2026
 
 
Document
 
Owner:
 
Customer
 
Operations
 
Team
 
 
Classification:
 
Internal
 
Support
 
Documentation

Chunk: 1
Metadata: {'source': 'customer_support.pdf', 'page': 2}
Characters: 987
Introduction
 
&
 
Customer
 
Support
 
Overview
 
1.
 
About
 
AcmeCloud
 
AcmeCloud
 
is
 
a
 
cloud-based
 
productivity
 
and
 
collaboration
 
platform
 
designed
 
for
 
individuals,
 
small
 
teams,
 
and
 
growing
 
organizations.
 
Customers
 
can
 
use
 
AcmeCloud
 
to
 
manage
 
projects,
 
collaborate
 
with
 
team
 
members,
 
store
 
files,
 
generate
 
reports,
 
and
 
integrate
 
their
 
existing
 
systems
 
through
 
the
 
AcmeCloud
 
API.
 
AcmeCloud
 
provides
 
four
 
subscription
 
plans:
 
●
 
Free
 
●
 
Starter
 
●
 
Professional
 
●
 
Business
 
Each
 
plan
 
provides
 
different
 

## 10. Check chunk sizes

This gives us a basic sanity check before embedding.

In [14]:
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print("Smallest chunk:", min(chunk_lengths))
print("Largest chunk:", max(chunk_lengths))
print("Average chunk:", sum(chunk_lengths) / len(chunk_lengths))

Smallest chunk: 224
Largest chunk: 1000
Average chunk: 753.9142857142857


## 11. Create the embedding model

Now we convert text into numerical vectors.

An embedding model maps text into a vector space where semantically similar text should be relatively close together.

For local learning we use a Sentence Transformer model. The model will download the first time it is used.

In [15]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

/home/workspace/learning/AI_Support_Eng/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2221.66it/s]


## 12. Understand what an embedding looks like

Before giving the embeddings to Chroma, let's generate one ourselves.

In [16]:
test_text = "Customers can request a refund within 14 days of purchase."

vector = embedding_model.embed_query(test_text)

print("Vector dimensions:", len(vector))
print("First 10 values:", vector[:10])

Vector dimensions: 384
First 10 values: [-0.07072871923446655, 0.036884959787130356, 0.08772902935743332, 0.0005379908252507448, -0.04109445586800575, -0.007893901318311691, -0.013301299884915352, 0.015027751214802265, 0.004488622769713402, -0.0012661400251090527]


The important idea is:

```text
"refund within 14 days"
          ↓
      embedding model
          ↓
[0.02, -0.13, 0.08, ...]
```

The vector itself is not human-readable. Its usefulness comes from comparing vectors mathematically.

## 13. Create the Chroma vector database

Chroma will store:

- the chunk text
- its embedding
- metadata

We use a persistent directory so the database survives after the notebook stops.

In [17]:
CHROMA_PATH.mkdir(parents=True, exist_ok=True)

collection_name = "acmecloud_support_handbook"

vectorstore = Chroma(
    collection_name=collection_name,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_PATH),
)

print("Chroma database:", CHROMA_PATH)
print("Collection:", collection_name)

Chroma database: /home/workspace/learning/AI_Support_Eng/data/chroma_db
Collection: acmecloud_support_handbook


## 14. Add our chunks to Chroma

This is the point where the chunks are embedded and stored in the vector database.

In [18]:
ids = [f"handbook-chunk-{index}" for index in range(len(chunks))]

vectorstore.add_documents(
    documents=chunks,
    ids=ids,
)

print(f"Added {len(chunks)} chunks to Chroma.")

Added 35 chunks to Chroma.


## 15. Verify the vector database

We should verify that the collection actually contains our chunks.

In [19]:
collection = vectorstore._collection

print("Stored documents:", collection.count())

Stored documents: 35


## 16. First semantic search

Now the actual retrieval part begins.

Notice that we are **not searching for an exact phrase**.

We will ask a natural-language question:

In [20]:
query = "How long do customers have to request a refund?"

results = vectorstore.similarity_search(
    query,
    k=4,
)

for index, result in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {index}")
    print("Metadata:", result.metadata)
    print(result.page_content)
    print()

Result 1
Metadata: {'source': 'customer_support.pdf', 'page': 8}
Refund
 
Policy
 
10.
 
Standard
 
Refund
 
Policy
 
AcmeCloud
 
offers
 
a
 
14-day
 
refund
 
period
 
for
 
eligible
 
purchases.
 
Customers
 
may
 
request
 
a
 
refund
 
within
 
14
 
days
 
of
 
the
 
original
 
purchase.
 
Annual
 
subscription
 
purchases
 
are
 
generally
 
eligible
 
for
 
a
 
full
 
refund
 
when
 
the
 
request
 
is
 
submitted
 
within
 
this
 
period.
 
Refund
 
eligibility
 
depends
 
on
 
the
 
transaction
 
type
 
and
 
the
 
circumstances
 
surrounding
 
the
 
request.
 
 
11.
 
Billing
 
Errors
 
Billing
 
errors
 
may
 
qualify
 
for
 
a
 
refund
 
even
 
when
 
the
 
normal
 
14-day
 
period
 
has
 
expired.
 
Examples
 
include:
 
●
 
Verified
 
duplicate
 
charges
 
●
 
Incorrect
 
subscription
 
charges
 
●
 
Charges
 
caused
 
by
 
a
 
confirmed
 
billing
 
system
 
error
 
A
 
support
 
engineer
 
should
 
verify
 
the
 
transaction
 
before
 
promising
 
a
 
refund.
 
If
 
the


## 17. Similarity search with scores

Scores are useful when debugging retrieval.

A lower distance generally means the retrieved vector is closer to the query vector for the distance metric being used. Do not blindly interpret the raw number as a probability.

In [21]:
results_with_scores = vectorstore.similarity_search_with_score(
    query,
    k=5,
)

for index, (document, score) in enumerate(results_with_scores, start=1):
    print("=" * 80)
    print(f"Result {index}")
    print("Distance:", score)
    print("Metadata:", document.metadata)
    print(document.page_content[:1000])

Result 1
Distance: 0.7334810495376587
Metadata: {'source': 'customer_support.pdf', 'page': 8}
Refund
 
Policy
 
10.
 
Standard
 
Refund
 
Policy
 
AcmeCloud
 
offers
 
a
 
14-day
 
refund
 
period
 
for
 
eligible
 
purchases.
 
Customers
 
may
 
request
 
a
 
refund
 
within
 
14
 
days
 
of
 
the
 
original
 
purchase.
 
Annual
 
subscription
 
purchases
 
are
 
generally
 
eligible
 
for
 
a
 
full
 
refund
 
when
 
the
 
request
 
is
 
submitted
 
within
 
this
 
period.
 
Refund
 
eligibility
 
depends
 
on
 
the
 
transaction
 
type
 
and
 
the
 
circumstances
 
surrounding
 
the
 
request.
 
 
11.
 
Billing
 
Errors
 
Billing
 
errors
 
may
 
qualify
 
for
 
a
 
refund
 
even
 
when
 
the
 
normal
 
14-day
 
period
 
has
 
expired.
 
Examples
 
include:
 
●
 
Verified
 
duplicate
 
charges
 
●
 
Incorrect
 
subscription
 
charges
 
●
 
Charges
 
caused
 
by
 
a
 
confirmed
 
billing
 
system
 
error
 
A
 
support
 
engineer
 
should
 
verify
 
the
 
transaction
 
before
 
promis

## 18. Test several realistic support questions

A single successful query proves very little. Retrieval should be tested with different kinds of questions.

In [22]:
test_queries = [
    "What is the refund period?",
    "Can I get my money back if I bought an annual plan five days ago?",
    "Does the Starter plan include API access?",
    "How many team members can a Professional account have?",
    "What should I do if my API returns HTTP 429?",
    "What should support do when a customer is charged twice?",
    "What information must never be requested from a customer?",
]

for query in test_queries:
    print("\n" + "#" * 100)
    print("QUERY:", query)

    results = vectorstore.similarity_search_with_score(query, k=2)

    for index, (document, score) in enumerate(results, start=1):
        print(f"\nResult {index} | distance={score:.4f} | metadata={document.metadata}")
        print(document.page_content[:700])


####################################################################################################
QUERY: What is the refund period?

Result 1 | distance=0.7314 | metadata={'page': 8, 'source': 'customer_support.pdf'}
Refund
 
Policy
 
10.
 
Standard
 
Refund
 
Policy
 
AcmeCloud
 
offers
 
a
 
14-day
 
refund
 
period
 
for
 
eligible
 
purchases.
 
Customers
 
may
 
request
 
a
 
refund
 
within
 
14
 
days
 
of
 
the
 
original
 
purchase.
 
Annual
 
subscription
 
purchases
 
are
 
generally
 
eligible
 
for
 
a
 
full
 
refund
 
when
 
the
 
request
 
is
 
submitted
 
within
 
this
 
period.
 
Refund
 
eligibility
 
depends
 
on
 
the
 
transaction
 
type
 
and
 
the
 
circumstances
 
surrounding
 
the
 
request.
 
 
11.
 
Billing
 
Errors
 
Billing
 
errors
 
may
 
qualify
 
for
 
a
 
refund
 
even
 
when
 
the
 
normal
 
14-day
 
period
 
has
 
expired.
 
Examples
 
include:
 
●
 
Verified
 
duplic

Result 2 | distance=0.7855 | metadata={'page': 9, 'source': 'customer_support

## 19. Try questions that RAG should NOT answer from the PDF

This is an important experiment for our final architecture.

Try questions such as:

- `What is my current subscription?`
- `Show me my last three invoices.`
- `Do I have any open support tickets?`
- `What did I purchase recently?`

The vector database may still return something because semantic search always tries to find the closest text.

That does **not** mean the answer exists in the document.

This is one of the core reasons our final Support Agent needs both **RAG and database tools**.

In [23]:
wrong_source_queries = [
    "What is my current subscription?",
    "Show me my last three invoices.",
    "Do I have any open support tickets?",
    "What did I purchase recently?",
]

for query in wrong_source_queries:
    print("\n" + "=" * 100)
    print("QUERY:", query)
    result = vectorstore.similarity_search(query, k=1)[0]
    print("Retrieved:", result.metadata)
    print(result.page_content[:700])


QUERY: What is my current subscription?
Retrieved: {'source': 'customer_support.pdf', 'page': 16}
been
 
exposed.
 
●
 
Unauthorized
 
access
 
is
 
suspected.
 
●
 
Suspicious
 
API
 
activity
 
is
 
detected.
 
 
Frequently
 
Asked
 
Questions
 
29.
 
Can
 
I
 
cancel
 
my
 
subscription?
 
Yes.
 
Customers
 
can
 
cancel
 
their
 
subscription
 
through
 
the
 
account
 
billing
 
settings.
 
Cancellation
 
normally
 
prevents
 
the
 
next
 
renewal.
 
It
 
does
 
not
 
necessarily
 
result
 
in
 
an
 
automatic
 
refund
 
for
 
the
 
current
 
billing
 
period.
 
Refund
 
eligibility
 
is
 
determined
 
separately
 
according
 
to
 
the
 
refund
 
policy.

QUERY: Show me my last three invoices.
Retrieved: {'source': 'customer_support.pdf', 'page': 6}
Invoices
 
contain:
 
●
 
Invoice
 
number
 
●
 
Customer
 
information
 
●
 
Subscription
 
plan
 
●
 
Amount
 
●
 
Issue
 
date
 
●
 
Due
 
date
 
●
 
Payment
 
status
 
Possible
 
invoice
 
statuses
 
include:
 
●
 
Paid
 
●
 
Pend

## 20. Create a LangChain retriever

So far we have used Chroma directly.

LangChain provides a retriever abstraction so the rest of our application doesn't need to know the exact vector database implementation.

In [24]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

retrieved_documents = retriever.invoke(
    "What happens if a customer is charged twice?"
)

for document in retrieved_documents:
    print("=" * 80)
    print(document.metadata)
    print(document.page_content)

{'page': 7, 'source': 'customer_support.pdf'}
customers.
 
 
9.
 
Duplicate
 
Charges
 
If
 
a
 
customer
 
believes
 
they
 
were
 
charged
 
twice,
 
support
 
should
 
first
 
verify
 
the
 
customer's
 
invoices
 
and
 
payment
 
records.
 
Possible
 
explanations
 
include:
 
●
 
One
 
charge
 
is
 
an
 
authorization
 
hold.
 
●
 
A
 
subscription
 
renewal
 
and
 
an
 
additional
 
purchase
 
occurred
 
close
 
together.
 
●
 
A
 
payment
 
was
 
retried
 
after
 
an
 
initial
 
failure.
 
●
 
A
 
genuine
 
duplicate
 
transaction
 
occurred.
 
If
 
two
 
successful
 
transactions
 
correspond
 
to
 
the
 
same
 
invoice
 
or
 
purchase,
 
the
 
issue
 
should
 
be
 
escalated
 
to
 
the
 
billing
 
team.
 
Customers
 
should
 
not
 
be
 
promised
 
an
 
immediate
 
refund
 
until
 
the
 
transaction
 
has
 
been
 
verified.
{'page': 17, 'source': 'customer_support.pdf'}
through
 
the
 
Billing
 
section
 
of
 
the
 
customer's
 
AcmeCloud
 
account.
 
Support
 
can
 
also
 
ret

## 21. Build the context that an LLM could consume

The retriever returns `Document` objects. An LLM generally needs a string containing the relevant context.

For now, we will manually build that context so the transformation is visible.

In [25]:
query = "What is the refund policy for an annual subscription?"

retrieved_documents = retriever.invoke(query)

context_parts = []

for document in retrieved_documents:
    source = document.metadata.get("source", "unknown")
    page = document.metadata.get("page", "unknown")

    context_parts.append(
        f"[Source: {source}, page {page}]\n"
        f"{document.page_content}"
    )

context = "\n\n---\n\n".join(context_parts)

print(context)

[Source: customer_support.pdf, page 8]
Refund
 
Policy
 
10.
 
Standard
 
Refund
 
Policy
 
AcmeCloud
 
offers
 
a
 
14-day
 
refund
 
period
 
for
 
eligible
 
purchases.
 
Customers
 
may
 
request
 
a
 
refund
 
within
 
14
 
days
 
of
 
the
 
original
 
purchase.
 
Annual
 
subscription
 
purchases
 
are
 
generally
 
eligible
 
for
 
a
 
full
 
refund
 
when
 
the
 
request
 
is
 
submitted
 
within
 
this
 
period.
 
Refund
 
eligibility
 
depends
 
on
 
the
 
transaction
 
type
 
and
 
the
 
circumstances
 
surrounding
 
the
 
request.
 
 
11.
 
Billing
 
Errors
 
Billing
 
errors
 
may
 
qualify
 
for
 
a
 
refund
 
even
 
when
 
the
 
normal
 
14-day
 
period
 
has
 
expired.
 
Examples
 
include:
 
●
 
Verified
 
duplicate
 
charges
 
●
 
Incorrect
 
subscription
 
charges
 
●
 
Charges
 
caused
 
by
 
a
 
confirmed
 
billing
 
system
 
error
 
A
 
support
 
engineer
 
should
 
verify
 
the
 
transaction
 
before
 
promising
 
a
 
refund.
 
If
 
the
 
issue
 
cannot
 
be
 
ve

## 22. What we have built

At this point the RAG subsystem is:

```text
support_handbook.pdf
        │
        ▼
    PDF reader
        │
        ▼
     Documents
        │
        ▼
  Text splitter
        │
        ▼
      Chunks
        │
        ▼
Embedding model
        │
        ▼
     Vectors
        │
        ▼
     ChromaDB
        │
        ▼
    Retriever
        │
        ▼
Relevant documents
        │
        ▼
     LLM later
```

We have intentionally stopped before adding the LLM.